# torch.backends — Performance Tuning Your PyTorch Setup

**Module 20** | Prerequisites: Module 07 (Training) | Time: ~2 hours

This notebook explores `torch.backends` — the configuration layer that controls hardware-specific optimizations in PyTorch. A single line of configuration can yield 2–10x speedups.

**What you'll learn:**
- All backend settings and what they control
- cuDNN benchmark mode and when to use it
- TF32 precision and its speed/accuracy tradeoff
- OpenMP threading for CPU parallelism
- opt_einsum strategies for optimized contractions
- Configuration presets for training, inference, and debugging

In [ ]:
import torch
import torch.nn as nn
import time

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## What Are torch.backends?

`torch.backends` is a configuration namespace that controls **how** PyTorch executes operations at the hardware level. It doesn't change what computation is performed — it changes which algorithm, precision, or parallelism strategy is used.

```
torch.backends
├── cudnn          # NVIDIA cuDNN (convolutions, RNNs)
├── cuda           # NVIDIA CUDA (matmul, SDPA)
├── mkldnn         # Intel oneDNN (CPU conv, linear)
├── mkl            # Intel MKL (BLAS/LAPACK)
├── openmp         # OpenMP (CPU threading)
├── opt_einsum     # Optimized einsum contraction
└── mps            # Apple Metal (M1/M2/M3)
```

All settings are **runtime toggles** — no recompilation needed.

## Print All Backend Settings

Let's start by inspecting the current state of all backends:

In [ ]:
print("=== cuDNN ===")
print(f"  enabled:        {torch.backends.cudnn.enabled}")
print(f"  benchmark:      {torch.backends.cudnn.benchmark}")
print(f"  deterministic:  {torch.backends.cudnn.deterministic}")
print(f"  allow_tf32:     {torch.backends.cudnn.allow_tf32}")
print(f"  is_available(): {torch.backends.cudnn.is_available()}")
print()
print("=== CUDA matmul ===")
print(f"  allow_tf32: {torch.backends.cuda.matmul.allow_tf32}")
print(f"  allow_fp16_reduced_precision_reduction: {torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction}")
print(f"  allow_bf16_reduced_precision_reduction: {torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction}")
print()
print("=== CUDA SDPA ===")
print(f"  flash_sdp_enabled:         {torch.backends.cuda.flash_sdp_enabled()}")
print(f"  mem_efficient_sdp_enabled:  {torch.backends.cuda.mem_efficient_sdp_enabled()}")
print(f"  math_sdp_enabled:           {torch.backends.cuda.math_sdp_enabled()}")
print()
print("=== MKL-DNN (oneDNN) ===")
print(f"  enabled:        {torch.backends.mkldnn.enabled}")
print(f"  is_available(): {torch.backends.mkldnn.is_available()}")
print()
print("=== OpenMP ===")
print(f"  num_threads: {torch.get_num_threads()}")
print()
print("=== opt_einsum ===")
print(f"  enabled:  {torch.backends.opt_einsum.enabled}")
if torch.backends.opt_einsum.enabled:
    print(f"  strategy: {torch.backends.opt_einsum.strategy}")
print()
print("=== MPS (Apple Metal) ===")
print(f"  is_available(): {torch.backends.mps.is_available()}")
print(f"  is_built():     {torch.backends.mps.is_built()}")
print()
print(f"=== float32_matmul_precision: {torch.get_float32_matmul_precision()} ===")

## cuDNN Settings

cuDNN is NVIDIA's library for deep neural network primitives (convolutions, RNNs, normalization).

### Key settings:

| Setting | Default | Purpose |
|---------|---------|--------|
| `enabled` | True | Use cuDNN at all |
| `benchmark` | False | Auto-tune convolution algorithms |
| `deterministic` | False | Force reproducible (but slower) algorithms |
| `allow_tf32` | True | Allow TF32 for convolutions on Ampere+ |

In [ ]:
# cuDNN benchmark mode demonstration
# When enabled, cuDNN tries multiple algorithm variants for each conv shape
# and caches the fastest one

print("cuDNN Benchmark Mode")
print("=" * 50)

# Show toggle behavior
print(f"\nDefault: torch.backends.cudnn.benchmark = {torch.backends.cudnn.benchmark}")

torch.backends.cudnn.benchmark = True
print(f"After setting True: {torch.backends.cudnn.benchmark}")

torch.backends.cudnn.benchmark = False
print(f"After setting False: {torch.backends.cudnn.benchmark}")

print("\nWhen to enable benchmark mode:")
print("  \u2705 Fixed input sizes (CNNs with constant batch/image size)")
print("  \u2705 Long training runs (amortize initial tuning cost)")
print("  \u274c Variable input sizes (NLP, object detection)")
print("  \u274c Short scripts (tuning cost > savings)")

In [ ]:
# Deterministic mode
print("cuDNN Deterministic Mode")
print("=" * 50)

print(f"\nDefault: torch.backends.cudnn.deterministic = {torch.backends.cudnn.deterministic}")

# Setting deterministic mode
torch.backends.cudnn.deterministic = True
print(f"Set to True: {torch.backends.cudnn.deterministic}")

# Reset
torch.backends.cudnn.deterministic = False

print("\nTradeoffs:")
print("  deterministic=False: Faster, but run-to-run variance")
print("  deterministic=True:  Slower (sometimes 2-3x), but bit-exact results")
print("\nFor full reproducibility, also call:")
print("  torch.use_deterministic_algorithms(True)")

## TF32 Precision

TF32 (TensorFloat-32) is a 19-bit floating point format on Ampere+ GPUs:

```
FP32:  1 sign + 8 exponent + 23 mantissa = 32 bits
TF32:  1 sign + 8 exponent + 10 mantissa = 19 bits
FP16:  1 sign + 5 exponent + 10 mantissa = 16 bits
BF16:  1 sign + 8 exponent +  7 mantissa = 16 bits
```

TF32 has the **range of FP32** with the **mantissa precision of FP16**. Tensor cores read FP32 inputs, round to TF32 internally, and accumulate in FP32.

**Result**: ~2-3x speedup with ~0.1% relative error for matmul/conv.

In [ ]:
# TF32 toggle demonstration
print("TF32 Precision Controls")
print("=" * 50)

print("\nFor convolutions (cuDNN):")
print(f"  torch.backends.cudnn.allow_tf32 = {torch.backends.cudnn.allow_tf32}")

print("\nFor matrix multiplication (cuBLAS):")
print(f"  torch.backends.cuda.matmul.allow_tf32 = {torch.backends.cuda.matmul.allow_tf32}")

# Toggle and show
print("\n--- Disabling TF32 (full FP32 precision) ---")
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
print(f"  cudnn.allow_tf32: {torch.backends.cudnn.allow_tf32}")
print(f"  cuda.matmul.allow_tf32: {torch.backends.cuda.matmul.allow_tf32}")

print("\n--- Enabling TF32 (faster, slight precision loss) ---")
torch.backends.cudnn.allow_tf32 = True
torch.backends.cuda.matmul.allow_tf32 = True
print(f"  cudnn.allow_tf32: {torch.backends.cudnn.allow_tf32}")
print(f"  cuda.matmul.allow_tf32: {torch.backends.cuda.matmul.allow_tf32}")

In [ ]:
# Precision comparison on CPU (showing the API; real TF32 effects require Ampere+ GPU)
print("Precision Impact Demo (CPU)")
print("=" * 50)

A = torch.randn(512, 512)
B = torch.randn(512, 512)

# On CPU, TF32 settings have no effect, but we demonstrate the API
torch.backends.cuda.matmul.allow_tf32 = False
result_fp32 = torch.mm(A, B)

torch.backends.cuda.matmul.allow_tf32 = True
result_tf32_setting = torch.mm(A, B)

# On CPU these are identical (TF32 only affects GPU tensor cores)
diff = (result_fp32 - result_tf32_setting).abs().max().item()
print(f"\nMax difference on CPU: {diff}")
print("(Expected 0.0 on CPU - TF32 only affects Ampere+ GPUs)")
print("\nOn A100 GPU, typical differences:")
print("  Max absolute diff:  ~1e-3")
print("  Mean relative error: ~1e-5")
print("  Speedup: 2-3x")

## torch.set_float32_matmul_precision()

High-level API that controls matmul precision globally:

| Level | Precision | Speed | Use Case |
|-------|-----------|-------|----------|
| `"highest"` | Full FP32 | Baseline | Debugging, validation |
| `"high"` | TF32 on Ampere+ | ~2-3x faster | Standard training |
| `"medium"` | TF32 + reduced accumulation | Fastest | Large-batch training |

In [ ]:
print("torch.set_float32_matmul_precision()")
print("=" * 50)

# Cycle through all levels
for level in ["highest", "high", "medium"]:
    torch.set_float32_matmul_precision(level)
    current = torch.get_float32_matmul_precision()
    print(f"  Set '{level}' -> get returns '{current}'")

# Set recommended default
torch.set_float32_matmul_precision("high")
print(f"\nRecommended for training: 'high'")
print(f"Current setting: {torch.get_float32_matmul_precision()}")

## OpenMP Threading

PyTorch uses OpenMP for CPU parallelism. The thread count affects all CPU operations: matmul, convolutions, element-wise ops.

**Guidelines:**
- Training: set to number of **physical** cores (not hyperthreads)
- Inference serving: reduce to allow concurrent requests
- DataLoader workers: reduce to avoid oversubscription

In [ ]:
print("OpenMP Thread Configuration")
print("=" * 50)

original = torch.get_num_threads()
print(f"\nCurrent thread count: {original}")

# Benchmark with different thread counts
A = torch.randn(1024, 1024)
B = torch.randn(1024, 1024)

print("\nTiming matmul (1024x1024) with different thread counts:")
for n in [1, 2, 4]:
    torch.set_num_threads(n)
    # Warmup
    _ = torch.mm(A, B)
    
    t0 = time.perf_counter()
    for _ in range(10):
        _ = torch.mm(A, B)
    elapsed = (time.perf_counter() - t0) * 1000
    print(f"  {n} thread(s): {elapsed:.1f}ms (10 iterations)")

# Restore
torch.set_num_threads(original)
print(f"\nRestored to {original} threads")

In [ ]:
# Environment variable approach (must be set BEFORE importing torch)
print("Environment Variable Approach")
print("=" * 50)
print()
print("Set BEFORE importing torch:")
print("  import os")
print("  os.environ['OMP_NUM_THREADS'] = '4'")
print("  os.environ['MKL_NUM_THREADS'] = '4'")
print("  import torch  # now uses 4 threads")
print()
print("Or from command line:")
print("  OMP_NUM_THREADS=4 python train.py")

## opt_einsum Strategy

For complex `torch.einsum` expressions with 3+ tensors, finding the optimal contraction order is NP-hard. `opt_einsum` provides heuristics that can be **orders of magnitude faster**.

| Strategy | Planning Speed | Result Quality | Use Case |
|----------|---------------|----------------|----------|
| `"greedy"` | Fast | Good | Default |
| `"optimal"` | Slow | Best | Small expressions |
| `"dp"` | Medium | Good | Balanced |
| `"auto"` | Adaptive | Best tradeoff | Recommended |

In [ ]:
print("opt_einsum Configuration")
print("=" * 50)

print(f"\nopt_einsum enabled: {torch.backends.opt_einsum.enabled}")

if torch.backends.opt_einsum.enabled:
    print(f"Current strategy: {torch.backends.opt_einsum.strategy}")
    
    # Multi-tensor contraction
    A = torch.randn(100, 50)
    B = torch.randn(50, 80)
    C = torch.randn(80, 100)
    D = torch.randn(100, 60)
    
    print("\nTiming 'ij,jk,kl,lm->im' with strategies:")
    for strategy in ["auto", "greedy", "optimal"]:
        torch.backends.opt_einsum.strategy = strategy
        _ = torch.einsum("ij,jk,kl,lm->im", A, B, C, D)  # warmup
        
        t0 = time.perf_counter()
        for _ in range(100):
            _ = torch.einsum("ij,jk,kl,lm->im", A, B, C, D)
        elapsed = (time.perf_counter() - t0) * 1000
        print(f"  strategy='{strategy}': {elapsed:.1f}ms (100 iters)")
    
    torch.backends.opt_einsum.strategy = "auto"
else:
    print("\nopt_einsum not available. Install with: pip install opt-einsum")

## Performance Checklist

### Training (Maximum Speed)

| Setting | Value | Why |
|---------|-------|-----|
| `cudnn.benchmark` | `True` | Auto-tune conv algorithms (fixed sizes) |
| `cudnn.deterministic` | `False` | Allow fastest non-deterministic algorithms |
| `cudnn.allow_tf32` | `True` | 2x conv speedup on Ampere+ |
| `cuda.matmul.allow_tf32` | `True` | 2-3x matmul speedup on Ampere+ |
| `set_float32_matmul_precision` | `"high"` | Enable TF32 via high-level API |

### Inference (Maximum Throughput)

| Setting | Value | Why |
|---------|-------|-----|
| All training settings | Same | Speed optimizations apply |
| `set_num_threads` | Physical cores | Match hardware |
| `torch.no_grad()` | Context | Skip gradient tracking |

### Debugging / Reproducibility

| Setting | Value | Why |
|---------|-------|-----|
| `cudnn.benchmark` | `False` | Avoid algorithm variation |
| `cudnn.deterministic` | `True` | Force deterministic algorithms |
| `allow_tf32` (both) | `False` | Full FP32 precision |
| `use_deterministic_algorithms` | `True` | Error on non-deterministic ops |
| `manual_seed(42)` | Fixed | Reproducible random state |

In [ ]:
# Quick-apply presets

def configure_training(fixed_sizes=True):
    """Optimal backend settings for training speed."""
    torch.backends.cudnn.benchmark = fixed_sizes
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
    print("Configured for training (maximum speed)")

def configure_inference(num_cores=4):
    """Optimal backend settings for inference throughput."""
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
    torch.set_num_threads(num_cores)
    print(f"Configured for inference ({num_cores} threads)")

def configure_reproducible(seed=42):
    """Optimal backend settings for reproducibility."""
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.set_float32_matmul_precision("highest")
    torch.use_deterministic_algorithms(True, warn_only=True)
    print(f"Configured for reproducibility (seed={seed})")

# Demo each preset
configure_training()
configure_inference(num_cores=4)
configure_reproducible(seed=123)

# Reset
torch.use_deterministic_algorithms(False)

## Backend flags() Context Manager

Temporarily override backend settings within a scope, with automatic restoration:

In [ ]:
print("torch.backends.flags() Context Manager")
print("=" * 50)

# Set baseline
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False

print(f"\nBefore: benchmark={torch.backends.cudnn.benchmark}, "
      f"deterministic={torch.backends.cudnn.deterministic}")

with torch.backends.flags(
    cudnn_benchmark=True,
    cudnn_deterministic=True,
):
    print(f"Inside: benchmark={torch.backends.cudnn.benchmark}, "
          f"deterministic={torch.backends.cudnn.deterministic}")

print(f"After:  benchmark={torch.backends.cudnn.benchmark}, "
      f"deterministic={torch.backends.cudnn.deterministic}")
print("\n(Settings automatically restored!)")

In [ ]:
# Practical use case: validation step with full precision
print("Use Case: Full-precision validation step")
print("=" * 50)

# Training uses TF32
torch.backends.cudnn.allow_tf32 = True
torch.backends.cuda.matmul.allow_tf32 = True
print(f"\nTraining mode: TF32 enabled")

# Temporarily disable for validation
with torch.backends.flags(allow_tf32=False):
    print(f"Validation: TF32 = {torch.backends.cudnn.allow_tf32} (disabled for precision)")
    # ... run validation ...

print(f"Back to training: TF32 = {torch.backends.cudnn.allow_tf32} (restored)")

## CUDA SDPA Backend Selection

Scaled Dot-Product Attention (SDPA) has multiple backend implementations. You can enable/disable each:

In [ ]:
print("SDPA Backend Control")
print("=" * 50)

print(f"\nFlash SDP:          {torch.backends.cuda.flash_sdp_enabled()}")
print(f"Memory-efficient SDP: {torch.backends.cuda.mem_efficient_sdp_enabled()}")
print(f"Math SDP:            {torch.backends.cuda.math_sdp_enabled()}")

# Disable flash attention (e.g., for debugging)
torch.backends.cuda.enable_flash_sdp(False)
print(f"\nAfter disabling Flash: {torch.backends.cuda.flash_sdp_enabled()}")

# Re-enable
torch.backends.cuda.enable_flash_sdp(True)
print(f"After re-enabling:     {torch.backends.cuda.flash_sdp_enabled()}")

print("\nNote: SDPA automatically picks the best available backend.")
print("Only disable backends for debugging or compatibility testing.")

## MKL-DNN and Intel CPU Optimization

In [ ]:
print("MKL-DNN (oneDNN) Status")
print("=" * 50)

print(f"\nMKL-DNN available: {torch.backends.mkldnn.is_available()}")
print(f"MKL-DNN enabled:   {torch.backends.mkldnn.enabled}")

if torch.backends.mkldnn.is_available():
    print("\noneDNN provides optimized CPU implementations for:")
    print("  - Convolutions (Conv1d, Conv2d, Conv3d)")
    print("  - Linear layers")
    print("  - Batch normalization")
    print("  - Pooling operations")
    print("\nPyTorch automatically routes to oneDNN when beneficial.")
    
    # Demo: check if a conv uses mkldnn format
    conv = nn.Conv2d(3, 64, 3, padding=1)
    x = torch.randn(1, 3, 32, 32)
    out = conv(x)
    print(f"\nConv2d output shape: {out.shape}")
    print(f"Uses MKL-DNN layout: {out.is_mkldnn}")

## MPS Backend (Apple Metal)

In [ ]:
print("MPS (Apple Metal) Backend")
print("=" * 50)

print(f"\nMPS available: {torch.backends.mps.is_available()}")
print(f"MPS built:     {torch.backends.mps.is_built()}")

if torch.backends.mps.is_available():
    device = torch.device("mps")
    x = torch.randn(1000, 1000, device=device)
    print(f"\nCreated tensor on MPS: {x.device}")
else:
    print("\nMPS requires:")
    print("  - Apple Silicon (M1/M2/M3/M4)")
    print("  - macOS 12.3+")
    print("  - PyTorch built with MPS support")
    print("\nUsage: tensor.to('mps') or torch.device('mps')")

## Comprehensive Settings Summary

In [ ]:
def print_all_settings():
    """Print a complete summary of all backend states."""
    settings = {
        "cudnn.enabled": torch.backends.cudnn.enabled,
        "cudnn.benchmark": torch.backends.cudnn.benchmark,
        "cudnn.deterministic": torch.backends.cudnn.deterministic,
        "cudnn.allow_tf32": torch.backends.cudnn.allow_tf32,
        "cuda.matmul.allow_tf32": torch.backends.cuda.matmul.allow_tf32,
        "cuda.flash_sdp": torch.backends.cuda.flash_sdp_enabled(),
        "mkldnn.enabled": torch.backends.mkldnn.enabled,
        "opt_einsum.enabled": torch.backends.opt_einsum.enabled,
        "num_threads": torch.get_num_threads(),
        "float32_matmul_precision": torch.get_float32_matmul_precision(),
    }
    
    print("\n" + "=" * 50)
    print("  COMPLETE BACKEND STATE")
    print("=" * 50)
    for k, v in settings.items():
        print(f"  {k:40s} = {v}")
    print("=" * 50)
    return settings

_ = print_all_settings()

## Exercise

**Task**: Write a function `configure_optimal(mode, **kwargs)` that accepts a mode string (`"train"`, `"infer"`, or `"debug"`) and configures all backends optimally for that use case.

Requirements:
- `"train"`: maximize speed (TF32, benchmark, non-deterministic)
- `"infer"`: maximize throughput (+ set thread count from kwargs)
- `"debug"`: maximize reproducibility (no TF32, deterministic, seed from kwargs)
- Return a dict of all settings that were applied

In [ ]:
# Exercise: implement configure_optimal

def configure_optimal(mode: str, **kwargs) -> dict:
    """Configure backends for the given mode.
    
    Args:
        mode: 'train', 'infer', or 'debug'
        **kwargs: 
            - num_threads (int): for infer mode
            - seed (int): for debug mode
            - fixed_sizes (bool): for train mode
    """
    settings = {}
    
    if mode == "train":
        fixed = kwargs.get("fixed_sizes", True)
        torch.backends.cudnn.benchmark = fixed
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
        settings = {
            "mode": "train",
            "cudnn.benchmark": fixed,
            "cudnn.deterministic": False,
            "allow_tf32": True,
            "precision": "high",
        }
    
    elif mode == "infer":
        n_threads = kwargs.get("num_threads", 4)
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
        torch.set_num_threads(n_threads)
        settings = {
            "mode": "infer",
            "cudnn.benchmark": True,
            "allow_tf32": True,
            "precision": "high",
            "num_threads": n_threads,
        }
    
    elif mode == "debug":
        seed = kwargs.get("seed", 42)
        torch.manual_seed(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.set_float32_matmul_precision("highest")
        torch.use_deterministic_algorithms(True, warn_only=True)
        settings = {
            "mode": "debug",
            "seed": seed,
            "cudnn.benchmark": False,
            "cudnn.deterministic": True,
            "allow_tf32": False,
            "precision": "highest",
            "deterministic_algorithms": True,
        }
    else:
        raise ValueError(f"Unknown mode: {mode}. Use 'train', 'infer', or 'debug'")
    
    return settings

# Test it
print("Training config:", configure_optimal("train"))
print("Inference config:", configure_optimal("infer", num_threads=8))
print("Debug config:", configure_optimal("debug", seed=123))

# Cleanup
torch.use_deterministic_algorithms(False)

## Key Takeaways

1. **`cudnn.benchmark = True`** — Auto-tune conv algorithms. Use with fixed input sizes.

2. **TF32 (`allow_tf32 = True`)** — 2-3x speedup on Ampere+ GPUs with minimal precision loss. Enabled by default in PyTorch 2.x.

3. **`set_float32_matmul_precision("high")`** — Clean API for enabling TF32. Set once at script start.

4. **`set_num_threads(N)`** — Match physical CPU cores. Over-subscribing hurts performance.

5. **`opt_einsum`** — Dramatically speeds up multi-tensor contractions. Install `opt-einsum` package.

6. **`backends.flags()`** — Scope backend settings temporarily. Great for validation steps.

7. **Determinism costs performance** — Only enable for debugging/CI. Never in production training.

---

**Next steps**: Apply these settings to your training scripts. A typical CNN training script can see 20-40% speedup just from `cudnn.benchmark + TF32`.